# Conduct empirical comparison

**Notebook contents:**
* Cleans the empirical fish dataset and extracts persistent trajectories
* Normalizes empirical position and velocity data
* Instantiates the D'Orsogna comparison model
* Computes and plots four radial profile comparisons between empirical and simulated data


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from matplotlib.ticker import FuncFormatter
from IPython.display import HTML

import matplotlib.pyplot as plt
from main import DorsognaTE

import json
import csv
import os

from scipy.spatial import cKDTree
from tqdm import tqdm
import calculateTE

from utils import set_plot_style

# Preliminaries

## Empirical dataset cleaning

We identify the longest contiguous window in which a minimum number of fish remain present across all frames, then export the retained trajectories for downstream analysis.

In [ ]:
min_persistent_fish = 40    
min_period_length = 100  

with open("empirical_data/schooling_frames.json", "r") as f:
    data = json.load(f)

frames = sorted(data.keys(), key=lambda x: int(x))

# set of IDs present in each frame
ids_by_frame = {fr: set(data[fr]["onfish"]) for fr in frames}

# find longest window
best = None

for i, start in enumerate(frames):
  core = ids_by_frame[start].copy()

  for j in range(i, len(frames)):
    fr = frames[j]
    core &= ids_by_frame[fr]  

    window_len = j - i + 1

    if len(core) >= min_persistent_fish and window_len >= min_period_length:
      if best is None or window_len > best["n_frames"]:
        best = {
          "start": int(start),
          "end": int(fr),
          "n_frames": window_len,
          "persistent_ids": sorted(core),
          "n_persistent": len(core)
        }

    if len(core) < min_persistent_fish:
        break

print(best)

#### Convert persistent fish_id's to CSV.

In [ ]:
with open("empirical_data/schooling_frames.json", "r") as f:
  data = json.load(f)

start = best["start"]
end = best["end"]
keep_ids = set(best["persistent_ids"])  

with open("empirical_data/schooling_persistent_ids.csv", "w", newline="") as f:
  writer = csv.writer(f)
  writer.writerow(["fish_id", "x", "y", "vx", "vy", "frame"])

  for fr in range(start, end + 1):
    d = data[str(fr)]

    for fish_id, x, y, vx, vy in zip(d["onfish"], d["px"], d["py"], d["vx"], d["vy"]):
      if fish_id in keep_ids:
        writer.writerow([fish_id, x, y, vx, vy, fr])

#### Normalize fish data

In [ ]:
df = pd.read_csv("empirical_data/schooling_persistent_ids.csv")

xmin, xmax = df["x"].min(), df["x"].max()
ymin, ymax = df["y"].min(), df["y"].max()

xscale = xmax - xmin
yscale = ymax - ymin

df["x"] = (df["x"] - xmin) / xscale
df["y"] = (df["y"] - ymin) / yscale
df["vx"] = df["vx"] / xscale
df["vy"] = df["vy"] / yscale

df.to_csv("empirical_data/schooling_persistent_ids_normalized.csv", index=False)

# Animate fish data

In [ ]:
fish_df = pd.read_csv("empirical_data/schooling_persistent_ids_normalized.csv").sort_values(["frame", "fish_id"])

frames = np.sort(fish_df["frame"].unique())
fish_ids = np.sort(fish_df["fish_id"].unique())

frame_xy = {}
frame_colors = {}

# align each frame to the same fish_id order
frame_data = {}
for fr, g in fish_df.groupby("frame"):
    g = g.set_index("fish_id").reindex(fish_ids)
    frame_data[fr] = g[["x", "y"]].to_numpy()

# settings
trail_length = 15     # how many past frames to show
interval_ms = 40      # lower = faster playback
skip = 2              # use every 2nd frame; bigger = faster

frames = frames[::skip]

# figure
fig, ax = plt.subplots(figsize=(7, 7))
ax.set_aspect("equal", adjustable="box")
ax.set_xlim(fish_df["x"].min() - 0.1, fish_df["x"].max() + 0.1)
ax.set_ylim(fish_df["y"].min() - 0.1, fish_df["y"].max() + 0.1)

xy0 = frame_data[frames[0]]

# current fish positions
scat = ax.scatter(xy0[:, 0], xy0[:, 1], s=20)

# one line per fish for trails
trail_lines = [ax.plot([], [], linewidth=1, alpha=0.5)[0] for _ in fish_ids]

title = ax.set_title(f"Golden Shiners")

def update(i):
    fr = frames[i]
    xy = frame_data[fr]
    scat.set_offsets(xy)
    title.set_text(f"Golden Shiners")

    # update each trail
    start_idx = max(0, i - trail_length + 1)
    trail_frames = frames[start_idx:i+1]

    for j, fid in enumerate(fish_ids):
        pts = np.array([frame_data[f][j] for f in trail_frames])
        trail_lines[j].set_data(pts[:, 0], pts[:, 1])

    return [scat, title, *trail_lines]

ani = FuncAnimation(
    fig,
    update,
    frames=len(frames),
    interval=interval_ms,
    blit=False
)

HTML(ani.to_jshtml())
ani.save("empirical_data/schooling_trailing.gif", writer="pillow", fps=30)

# Comparing radial profiles

## D'Orsogna model setup

Three different D'Orsogna simulation setups are instantiated for comparison with the empirical dataset.

In [ ]:

# Model settings
nbins = 25
dor_start_idx = 200
dor_end_idx = 300
use_com_relative_velocity = True

# List of setups to run
setups = [
    {"label": "single mill (3.0, 0.1)", "C": 3.0, "l": 0.1},
    {"label": "single mill (2.0, 0.1)", "C": 2.0, "l": 0.1},
    {"label": "single mill (0.9, 0.1)", "C": 0.9, "l": 0.1},
]

# common simulation settings
particle_count = 40
t_max = 40
vel_scale = 0.1
fps = 20
dt = 0.1
phenotype_name = "single_mill" 

# empirical fish data
fish_df = pd.read_csv("empirical_data/schooling_persistent_ids_normalized.csv")

In [ ]:
models = []

# run setups
for s in setups:
    model = DorsognaTE(outdir='empirical_data/TE_vals')
    model.develop_model(
        C=s["C"],
        l=s["l"],
        phenotype_name=phenotype_name,
        particle_count=particle_count,
        t_max=t_max,
        vel_scale=vel_scale,
        fps=fps,
        dt=dt
    )
    models.append({
        "label": s["label"],
        "C": s["C"],
        "l": s["l"],
        "model": model
    })

## I. Average radial density

### Helper functions

In [ ]:
def avg_density_profile_from_pos(pos, bins=None, nbins=25, frame_slice=None):
    """
    pos: array of shape (T, N, 2)
    bins: optional radial bin edges
    nbins: used only if bins is None
    frame_slice: e.g. slice(200, None) to keep only later frames

    Returns:
      profile_df with columns ['r_mid', 'density']
      bins used
    """
    if frame_slice is not None:
        pos = pos[frame_slice]

    T, N, _ = pos.shape

    # center of mass per frame
    com = pos.mean(axis=1, keepdims=True)              # (T,1,2)
    rvec = pos - com
    r = np.linalg.norm(rvec, axis=2)                   # (T,N)

    # choose bins if not supplied
    if bins is None:
        rmax = r.max()
        bins = np.linspace(0, rmax, nbins + 1)

    shell_areas = np.pi * (bins[1:]**2 - bins[:-1]**2)
    density_acc = np.zeros(len(bins) - 1)

    for t in range(T):
        counts, _ = np.histogram(r[t], bins=bins)
        density_t = counts / shell_areas
        density_acc += density_t

    density_avg = density_acc / T
    r_mid = 0.5 * (bins[:-1] + bins[1:])

    profile_df = pd.DataFrame({
        "r_mid": r_mid,
        "density": density_avg
    })
    return profile_df, bins

def avg_density_profile_from_fish_df(df, bins=None, nbins=25, frame_col="frame"):
    """
    df must have columns: frame, x, y
    bins: optional radial bin edges
    nbins: used only if bins is None

    Returns:
      profile_df with columns ['r_mid', 'density']
      bins used
    """
    df = df.sort_values([frame_col])

    # first pass: get all radii, so we can set common bins if not supplied
    all_radii = []
    grouped = list(df.groupby(frame_col))

    for fr, g in grouped:
        x = g["x"].to_numpy()
        y = g["y"].to_numpy()

        x_com = np.mean(x)
        y_com = np.mean(y)

        r = np.sqrt((x - x_com)**2 + (y - y_com)**2)
        all_radii.append(r)

    all_radii = np.concatenate(all_radii)

    if bins is None:
        rmax = all_radii.max()
        bins = np.linspace(0, rmax, nbins + 1)

    shell_areas = np.pi * (bins[1:]**2 - bins[:-1]**2)
    density_acc = np.zeros(len(bins) - 1)
    T = len(grouped)

    for fr, g in grouped:
        x = g["x"].to_numpy()
        y = g["y"].to_numpy()

        x_com = np.mean(x)
        y_com = np.mean(y)

        r = np.sqrt((x - x_com)**2 + (y - y_com)**2)

        counts, _ = np.histogram(r, bins=bins)
        density_t = counts / shell_areas
        density_acc += density_t

    density_avg = density_acc / T
    r_mid = 0.5 * (bins[:-1] + bins[1:])

    profile_df = pd.DataFrame({
        "r_mid": r_mid,
        "density": density_avg
    })
    return profile_df, bins

# compute max radii from raw data to get shared bins
def max_radius_pos(pos):
    com = pos.mean(axis=1, keepdims=True)
    r = np.linalg.norm(pos - com, axis=2)
    return r.max()

def max_radius_fish(df):
    max_r = 0
    for fr, g in df.groupby("frame"):
        x = g["x"].to_numpy()
        y = g["y"].to_numpy()
        x_com = np.mean(x)
        y_com = np.mean(y)
        r = np.sqrt((x - x_com)**2 + (y - y_com)**2)
        max_r = max(max_r, r.max())
    return max_r

### Shared radial bins

To ensure comparability, the empirical trajectories and all simulated trajectories are evaluated on a common set of radial bins derived from the maximum observed radius across datasets.

In [ ]:
rmax_candidates = [max_radius_fish(fish_df)]

for m in models:
    pos_window = m["model"].pos[dor_start_idx:dor_end_idx + 1]
    rmax_candidates.append(max_radius_pos(pos_window))

rmax_shared = max(rmax_candidates)
shared_bins = np.linspace(0, rmax_shared, nbins + 1)

### Average radial densities 

In [ ]:
# Empirical dataset density
fish_density, _ = avg_density_profile_from_fish_df(
    fish_df,
    bins=shared_bins
)

# D'Orsogna simulation densities
density_results = []
for m in models:
    pos_window = m["model"].pos[dor_start_idx:dor_end_idx + 1]
    dor_density, _ = avg_density_profile_from_pos(
        pos_window,
        bins=shared_bins
    )
    density_results.append({
        "label": m["label"],
        "df": dor_density
    })

### Visualize profiles

In [ ]:
set_plot_style()

plt.figure(figsize=(8, 4)) 
plt.figure(figsize=(8, 5))
plt.plot(fish_density["r_mid"], fish_density["density"], linewidth=3, label="Golden shiners")

for res in density_results:
    plt.plot(res["df"]["r_mid"], res["df"]["density"], label=res["label"])

plt.axhline(0, color="gray", linewidth=0.6, linestyle=":")
plt.xlabel("Distance from Swarm Center ($r$)")
plt.ylabel("Averaged Radial Density")
plt.title("Averaged Radial Density Profile")

# bounds
plt.grid()

plt.title("Average radial density profile")
 
# legend
plt.legend(fontsize=9, frameon=False)
plt.tight_layout()


# save
folder_name = "empirical_graphs"
file_name = f"rad_den.png"

if not os.path.exists(folder_name):
    os.makedirs(folder_name)
    print(f"folder {folder_name} created")

save_path = os.path.join(folder_name, file_name)

plt.savefig(save_path, dpi=300, bbox_inches="tight")

plt.show()

## II. Average radial momentum

### Helper functions

In [ ]:
def avg_radial_momentum_from_pos_vel(pos, vel, bins=None, nbins=25, frame_slice=None,
                                     com_relative_velocity=True, eps=1e-12):
    if frame_slice is not None:
        pos = pos[frame_slice]
        vel = vel[frame_slice]

    T, N, _ = pos.shape

    com = pos.mean(axis=1, keepdims=True)
    rvec = pos - com
    r = np.linalg.norm(rvec, axis=2)

    rhat = rvec / np.maximum(r[..., None], eps)

    if com_relative_velocity:
        v_com = vel.mean(axis=1, keepdims=True)
        vel_use = vel - v_com
    else:
        vel_use = vel

    u_rad = np.sum(vel_use * rhat, axis=2)

    if bins is None:
        rmax = r.max()
        bins = np.linspace(0, rmax, nbins + 1)

    shell_areas = np.pi * (bins[1:]**2 - bins[:-1]**2)
    prad_acc = np.zeros(len(bins) - 1)

    for t in range(T):
        sum_urad, _ = np.histogram(r[t], bins=bins, weights=u_rad[t])
        prad_t = sum_urad / shell_areas
        prad_acc += prad_t

    prad_avg = prad_acc / T
    r_mid = 0.5 * (bins[:-1] + bins[1:])

    return pd.DataFrame({
        "r_mid": r_mid,
        "prad": prad_avg
    }), bins

def avg_radial_momentum_from_fish_df(df, bins=None, nbins=25, frame_col="frame",
                                     com_relative_velocity=True, eps=1e-12):
    df = df.sort_values(frame_col)
    grouped = list(df.groupby(frame_col))

    all_radii = []
    for fr, g in grouped:
        x = g["x"].to_numpy()
        y = g["y"].to_numpy()
        x_com = np.mean(x)
        y_com = np.mean(y)
        r = np.sqrt((x - x_com)**2 + (y - y_com)**2)
        all_radii.append(r)

    all_radii = np.concatenate(all_radii) if len(all_radii) else np.array([0.0])

    if bins is None:
        rmax = all_radii.max()
        bins = np.linspace(0, rmax, nbins + 1)

    shell_areas = np.pi * (bins[1:]**2 - bins[:-1]**2)
    prad_acc = np.zeros(len(bins) - 1)
    T = len(grouped)

    for fr, g in grouped:
        x = g["x"].to_numpy()
        y = g["y"].to_numpy()
        vx = g["vx"].to_numpy()
        vy = g["vy"].to_numpy()

        x_com = np.mean(x)
        y_com = np.mean(y)

        rx = x - x_com
        ry = y - y_com
        r = np.sqrt(rx**2 + ry**2)

        rxhat = rx / np.maximum(r, eps)
        ryhat = ry / np.maximum(r, eps)

        if com_relative_velocity:
            vx = vx - np.mean(vx)
            vy = vy - np.mean(vy)

        u_rad = vx * rxhat + vy * ryhat

        sum_urad, _ = np.histogram(r, bins=bins, weights=u_rad)
        prad_t = sum_urad / shell_areas
        prad_acc += prad_t

    prad_avg = prad_acc / T
    r_mid = 0.5 * (bins[:-1] + bins[1:])

    return pd.DataFrame({
        "r_mid": r_mid,
        "prad": prad_avg
    }), bins

### Average radial momentum

In [ ]:
# Empirical dataset radial momentum
fish_prad, _ = avg_radial_momentum_from_fish_df(
    fish_df,
    bins=shared_bins,
    com_relative_velocity=use_com_relative_velocity
)

# D'Orsogna simulation radial momentum
prad_results = []
for m in models:
    pos_window = m["model"].pos[dor_start_idx:dor_end_idx + 1]
    vel_window = m["model"].vel[dor_start_idx:dor_end_idx + 1]

    dor_prad, _ = avg_radial_momentum_from_pos_vel(
        pos_window,
        vel_window,
        bins=shared_bins,
        com_relative_velocity=use_com_relative_velocity
    )

    prad_results.append({
        "label": m["label"],
        "df": dor_prad
    })

### Visualize profiles

In [ ]:
set_plot_style()

plt.figure(figsize=(8, 4)) 
plt.figure(figsize=(8, 5))
plt.plot(fish_prad["r_mid"], fish_prad["prad"], linewidth=3, label="Golden shiners")

for res in prad_results:
    plt.plot(res["df"]["r_mid"], res["df"]["prad"], label=res["label"])

plt.axhline(0, color="gray", linewidth=0.6, linestyle=":")
plt.xlabel("Distance from Swarm Center ($r$)")
plt.ylabel("Averaged Radial Momentum")
plt.title("Averaged Radial Momentum Profile")
 
# legend
plt.legend(fontsize=9, frameon=False)
plt.tight_layout()


# save
folder_name = "empirical_graphs"
file_name = f"rad_mom.png"

if not os.path.exists(folder_name):
    os.makedirs(folder_name)
    print(f"folder {folder_name} created")

save_path = os.path.join(folder_name, file_name)

plt.savefig(save_path, dpi=300, bbox_inches="tight")

plt.show()

## Average tangential momentum 

### Helper functions

In [ ]:
def avg_tangential_momentum_from_pos_vel(pos, vel, bins=None, nbins=25,
                                         com_relative_velocity=True, eps=1e-12):

    T, N, _ = pos.shape

    # center of mass position
    com = pos.mean(axis=1, keepdims=True)
    rvec = pos - com
    r = np.linalg.norm(rvec, axis=2)

    # unit radial vector
    rhat = rvec / np.maximum(r[..., None], eps)

    # unit tangential vector: rotate radial by +90 degrees
    that = np.empty_like(rhat)
    that[..., 0] = -rhat[..., 1]
    that[..., 1] =  rhat[..., 0]

    # optionally subtract COM velocity
    if com_relative_velocity:
        v_com = vel.mean(axis=1, keepdims=True)
        vel_use = vel - v_com
    else:
        vel_use = vel

    # tangential velocity component
    u_tang = np.sum(vel_use * that, axis=2)

    # bins
    if bins is None:
        rmax = r.max()
        bins = np.linspace(0, rmax, nbins + 1)

    shell_areas = np.pi * (bins[1:]**2 - bins[:-1]**2)
    ptang_acc = np.zeros(len(bins) - 1)

    for t in range(T):
        sum_utang, _ = np.histogram(r[t], bins=bins, weights=u_tang[t])
        ptang_t = sum_utang / shell_areas
        ptang_acc += ptang_t

    ptang_avg = ptang_acc / T
    r_mid = 0.5 * (bins[:-1] + bins[1:])

    return pd.DataFrame({
        "r_mid": r_mid,
        "ptang": ptang_avg
    }), bins

def avg_tangential_momentum_from_fish_df(df, bins=None, nbins=25,
                                         com_relative_velocity=True, eps=1e-12):
    """
    df must have columns: frame, x, y, vx, vy
    Returns DataFrame with columns: r_mid, ptang
    """
    df = df.sort_values("frame")
    grouped = list(df.groupby("frame"))

    # choose bins if not supplied
    all_radii = []
    for fr, g in grouped:
        x = g["x"].to_numpy()
        y = g["y"].to_numpy()
        x_com = np.mean(x)
        y_com = np.mean(y)
        r = np.sqrt((x - x_com)**2 + (y - y_com)**2)
        all_radii.append(r)

    all_radii = np.concatenate(all_radii) if len(all_radii) else np.array([0.0])

    if bins is None:
        rmax = all_radii.max()
        bins = np.linspace(0, rmax, nbins + 1)

    shell_areas = np.pi * (bins[1:]**2 - bins[:-1]**2)
    ptang_acc = np.zeros(len(bins) - 1)
    T = len(grouped)

    for fr, g in grouped:
        x = g["x"].to_numpy()
        y = g["y"].to_numpy()
        vx = g["vx"].to_numpy()
        vy = g["vy"].to_numpy()

        x_com = np.mean(x)
        y_com = np.mean(y)

        rx = x - x_com
        ry = y - y_com
        r = np.sqrt(rx**2 + ry**2)

        # unit radial vector
        rxhat = rx / np.maximum(r, eps)
        ryhat = ry / np.maximum(r, eps)

        # tangential unit vector
        txhat = -ryhat
        tyhat =  rxhat

        # optionally subtract COM velocity
        if com_relative_velocity:
            vx = vx - np.mean(vx)
            vy = vy - np.mean(vy)

        # tangential velocity
        u_tang = vx * txhat + vy * tyhat

        sum_utang, _ = np.histogram(r, bins=bins, weights=u_tang)
        ptang_t = sum_utang / shell_areas
        ptang_acc += ptang_t

    ptang_avg = ptang_acc / T
    r_mid = 0.5 * (bins[:-1] + bins[1:])

    return pd.DataFrame({
        "r_mid": r_mid,
        "ptang": ptang_avg
    }), bins


### Average tangential momentum

In [ ]:
# Empirical dataset tangential momentum
fish_ptang, _ = avg_tangential_momentum_from_fish_df(
    fish_df,
    bins=shared_bins,
    com_relative_velocity=use_com_relative_velocity
)

# D'Orsogna simulation tangential momentum
ptang_results = []
for m in models:
    pos_window = m["model"].pos[dor_start_idx:dor_end_idx + 1]
    vel_window = m["model"].vel[dor_start_idx:dor_end_idx + 1]

    dor_ptang, _ = avg_tangential_momentum_from_pos_vel(
        pos_window,
        vel_window,
        bins=shared_bins,
        com_relative_velocity=use_com_relative_velocity
    )

    ptang_results.append({
        "label": m["label"],
        "df": dor_ptang
    })

### Visualize profiles

In [ ]:
set_plot_style()

plt.figure(figsize=(8, 4)) 
plt.figure(figsize=(8, 5))
plt.plot(fish_ptang["r_mid"], fish_ptang["ptang"], linewidth=3, label="Golden shiners")

for res in ptang_results:
    plt.plot(res["df"]["r_mid"], res["df"]["ptang"], label=res["label"])

plt.axhline(0, color="gray", linewidth=0.6, linestyle=":")
plt.xlabel("Distance from Swarm Center ($r$)")
plt.ylabel("Averaged Tangential Momentum")
plt.title("Averaged Tangential Momentum Profile")
 
# legend
plt.legend(fontsize=9, frameon=False)
plt.tight_layout()


# save
folder_name = "empirical_graphs"
file_name = f"tan_mom.png"

if not os.path.exists(folder_name):
    os.makedirs(folder_name)
    print(f"folder {folder_name} created")

save_path = os.path.join(folder_name, file_name)

plt.savefig(save_path, dpi=300, bbox_inches="tight")

plt.show()

## IV. Average tangential velocity

### Helper functions

In [ ]:
def avg_tangential_velocity_from_pos_vel(pos, vel, bins=None, nbins=25,
                                         com_relative_velocity=True, eps=1e-12):
    """
    pos: (T, N, 2)
    vel: (T, N, 2)
    Returns DataFrame with columns: r_mid, vtang
    """
    T, N, _ = pos.shape

    # center of mass position
    com = pos.mean(axis=1, keepdims=True)
    rvec = pos - com
    r = np.linalg.norm(rvec, axis=2)

    # unit radial vector
    rhat = rvec / np.maximum(r[..., None], eps)

    # unit tangential vector
    that = np.empty_like(rhat)
    that[..., 0] = -rhat[..., 1]
    that[..., 1] =  rhat[..., 0]

    # optionally subtract COM velocity
    if com_relative_velocity:
        v_com = vel.mean(axis=1, keepdims=True)
        vel_use = vel - v_com
    else:
        vel_use = vel

    # tangential velocity component
    u_tang = np.sum(vel_use * that, axis=2)

    # bins
    if bins is None:
        rmax = r.max()
        bins = np.linspace(0, rmax, nbins + 1)

    vtang_acc = np.zeros(len(bins) - 1)

    for t in range(T):
        sum_utang, _ = np.histogram(r[t], bins=bins, weights=u_tang[t])
        counts, _ = np.histogram(r[t], bins=bins)

        # mean tangential velocity in each shell
        vtang_t = np.divide(
            sum_utang,
            counts,
            out=np.zeros_like(sum_utang, dtype=float),
            where=counts > 0
        )
        vtang_acc += vtang_t

    vtang_avg = vtang_acc / T
    r_mid = 0.5 * (bins[:-1] + bins[1:])

    return pd.DataFrame({
        "r_mid": r_mid,
        "vtang": vtang_avg
    }), bins

def avg_tangential_velocity_from_fish_df(df, bins=None, nbins=25,
                                         com_relative_velocity=True, eps=1e-12):
    """
    df must have columns: frame, x, y, vx, vy
    Returns DataFrame with columns: r_mid, vtang
    """
    df = df.sort_values("frame")
    grouped = list(df.groupby("frame"))

    # choose bins if not supplied
    all_radii = []
    for fr, g in grouped:
        x = g["x"].to_numpy()
        y = g["y"].to_numpy()
        x_com = np.mean(x)
        y_com = np.mean(y)
        r = np.sqrt((x - x_com)**2 + (y - y_com)**2)
        all_radii.append(r)

    all_radii = np.concatenate(all_radii) if len(all_radii) else np.array([0.0])

    if bins is None:
        rmax = all_radii.max()
        bins = np.linspace(0, rmax, nbins + 1)

    vtang_acc = np.zeros(len(bins) - 1)
    T = len(grouped)

    for fr, g in grouped:
        x = g["x"].to_numpy()
        y = g["y"].to_numpy()
        vx = g["vx"].to_numpy()
        vy = g["vy"].to_numpy()

        x_com = np.mean(x)
        y_com = np.mean(y)

        rx = x - x_com
        ry = y - y_com
        r = np.sqrt(rx**2 + ry**2)

        # unit radial vector
        rxhat = rx / np.maximum(r, eps)
        ryhat = ry / np.maximum(r, eps)

        # tangential unit vector
        txhat = -ryhat
        tyhat =  rxhat

        # optionally subtract COM velocity
        if com_relative_velocity:
            vx = vx - np.mean(vx)
            vy = vy - np.mean(vy)

        # tangential velocity component
        u_tang = vx * txhat + vy * tyhat

        sum_utang, _ = np.histogram(r, bins=bins, weights=u_tang)
        counts, _ = np.histogram(r, bins=bins)

        vtang_t = np.divide(
            sum_utang,
            counts,
            out=np.zeros_like(sum_utang, dtype=float),
            where=counts > 0
        )
        vtang_acc += vtang_t

    vtang_avg = vtang_acc / T
    r_mid = 0.5 * (bins[:-1] + bins[1:])

    return pd.DataFrame({
        "r_mid": r_mid,
        "vtang": vtang_avg
    }), bins

### Average tangential velocity

In [ ]:
# empirical tangential velocity
fish_vtang, _ = avg_tangential_velocity_from_fish_df(
    fish_df,
    bins=shared_bins,
    com_relative_velocity=use_com_relative_velocity
)

# model tangential velocity
vtang_results = []
for m in models:
    pos_window = m["model"].pos[dor_start_idx:dor_end_idx + 1]
    vel_window = m["model"].vel[dor_start_idx:dor_end_idx + 1]

    dor_vtang, _ = avg_tangential_velocity_from_pos_vel(
        pos_window,
        vel_window,
        bins=shared_bins,
        com_relative_velocity=use_com_relative_velocity
    )

    vtang_results.append({
        "label": m["label"],
        "df": dor_vtang
    })

### Visualize profiles

In [ ]:
set_plot_style()

plt.figure(figsize=(8, 4)) 
plt.figure(figsize=(8, 5))
plt.plot(fish_vtang["r_mid"], fish_vtang["vtang"], linewidth=3, label="Golden shiners")

for res in vtang_results:
    plt.plot(res["df"]["r_mid"], res["df"]["vtang"], label=res["label"])

plt.axhline(0, color="gray", linewidth=0.6, linestyle=":")
plt.xlabel("Distance from Swarm Center ($r$)")
plt.ylabel("Averaged Tangential Velocity")
plt.title("Averaged Tangential Velocity Profile")
 
# legend
plt.legend(fontsize=9, frameon=False)
plt.tight_layout()


# save
folder_name = "empirical_graphs"
file_name = f"tan_vel.png"

if not os.path.exists(folder_name):
    os.makedirs(folder_name)
    print(f"folder {folder_name} created")

save_path = os.path.join(folder_name, file_name)

plt.savefig(save_path, dpi=300, bbox_inches="tight")

plt.show()

# Comparing TE

In [ ]:
# settings
TE_embedding = 15
smooth_window = 5   

# time windows for each simulated setup
models[0]["t_start"], models[0]["t_end"] = 300,400
models[1]["t_start"], models[1]["t_end"] = 160, 260
models[2]["t_start"], models[2]["t_end"] = 270, 370

# empirical windows
empirical_ver2_start, empirical_ver2_end = 0, 350
empirical_ver4_start, empirical_ver4_end = 0, 350



## Helper functions

In [ ]:
def calculateTE_ver(pos, vel, TE_ver, i, j, TE_embedding):
  if TE_ver == 'ver2': # calculate TE using velocity
      te_vals = calculateTE.TEv2_JIDTKSG_vel(vel=vel, i=i, j=j, k=TE_embedding)
  elif TE_ver == 'ver3': # calculate TE using position
      te_vals = calculateTE.TEv3_JIDTKSG_pos(pos=pos, i=i, j=j, k=TE_embedding)
  elif TE_ver == 'ver4': # calculate TE using lizier method
      te_vals = calculateTE.TEv4_JIDTKSG_lizier(vel=vel, i=i, j=j, k=TE_embedding)
  else:
      raise ValueError(f"TE_ver: {TE_ver} not valid.")
  return te_vals

def calculateTE_ver(pos, vel, TE_ver, i, j, TE_embedding):
    if TE_ver == 'ver2':
        te_vals = calculateTE.TEv2_JIDTKSG_vel(vel=vel, i=i, j=j, k=TE_embedding)
    elif TE_ver == 'ver3':
        te_vals = calculateTE.TEv3_JIDTKSG_pos(pos=pos, i=i, j=j, k=TE_embedding)
    elif TE_ver == 'ver4':
        te_vals = calculateTE.TEv4_JIDTKSG_lizier(vel=vel, i=i, j=j, k=TE_embedding)
    else:
        raise ValueError(f"TE_ver: {TE_ver} not valid.")
    return te_vals

def compute_te(pos, vel, TE_ver, TE_embedding=15, nearest_neighbors=None, neighbor_radius=None,
               permute_seed=None, save_csv_path=None, save_mask_path=None):

    if neighbor_radius is not None and nearest_neighbors is not None:
        raise ValueError("Choose only one of fixed radius or nearest neighbors")

    T, N, _ = pos.shape
    pairwise_arr = [f"{i}-{j}" for i in range(N) for j in range(N) if i != j]

    TE_log_df = pd.DataFrame(index=range(T - 1), columns=pairwise_arr, dtype=float)
    mask_df = pd.DataFrame(index=range(T - 1), columns=pairwise_arr, dtype=float)

    vel_used = vel.copy()

    if permute_seed is not None:
        np.random.seed(permute_seed)
        permuted = np.empty_like(vel_used)
        for p in range(N):
            perm = np.random.permutation(T)
            permuted[:, p, :] = vel_used[perm, p, :]
        vel_used = permuted

    for i in tqdm(range(N), desc="Computing TE for each particle"):
        for j in range(N):
            if i != j:
                te_vals = calculateTE_ver(pos, vel_used, TE_ver, i, j, TE_embedding)
                TE_log_df[f"{i}-{j}"] = np.ravel(te_vals)[1:]

    if neighbor_radius is not None:
        for ts in tqdm(range(T - 1), desc="Masking by fixed radius"):
            tree = cKDTree(pos[ts])
            for i in range(N):
                neighbors = tree.query_ball_point(pos[ts][i], r=neighbor_radius)
                for j in neighbors:
                    if j != i:
                        mask_df.at[ts, f"{i}-{j}"] = 1.0

    elif nearest_neighbors is not None:
        for ts in tqdm(range(T - 1), desc="Determining nearest neighbor pairs"):
            tree = cKDTree(pos[ts])
            for i in range(N):
                distances, indices = tree.query(pos[ts][i], k=nearest_neighbors + 1)
                indices = np.atleast_1d(indices)
                for j in indices:
                    if j != i:
                        mask_df.at[ts, f"{i}-{j}"] = 1.0

    if neighbor_radius is not None or nearest_neighbors is not None:
        TE_log_df = TE_log_df * mask_df.values

    overall_avg_TE = TE_log_df.mean(axis=1)

    if save_csv_path is not None:
        TE_log_df.to_csv(save_csv_path, index=True)

    if save_mask_path is not None and (neighbor_radius is not None or nearest_neighbors is not None):
        mask_df.to_csv(save_mask_path, index=True)

    return TE_log_df, overall_avg_TE, mask_df

## Compute TE for empirical dataset

In [ ]:
# Load fish dataset
df = pd.read_csv("empirical_data/schooling_persistent_ids.csv")

# Sort properly
df = df.sort_values(["frame", "fish_id"]).copy()
df = df.head(574*40)

# Get ordered frames and fish IDs
frames = np.sort(df["frame"].unique())
fish_ids = np.sort(df["fish_id"].unique())

T = len(frames)
N = len(fish_ids)

print("T =", T)
print("N =", N)

# Build position and velocity tensors
pos = np.zeros((T, N, 2), dtype=float)
vel = np.zeros((T, N, 2), dtype=float)

for t, frame in enumerate(frames):
    sub = df[df["frame"] == frame].set_index("fish_id").loc[fish_ids]
    
    pos[t, :, 0] = sub["x"].to_numpy()
    pos[t, :, 1] = sub["y"].to_numpy()
    
    vel[t, :, 0] = sub["vx"].to_numpy()
    vel[t, :, 1] = sub["vy"].to_numpy()

print("pos shape:", pos.shape)
print("vel shape:", vel.shape)

In [ ]:
# Compute TE for ver2
compute_te(
    pos=pos,
    vel=vel,
    TE_ver="ver2",
    TE_embedding=15,
    save_csv_path="empirical_data/TE_vals/fish_TElog_ver2_k15_full.csv"
)

In [ ]:
# Compute TE for ver4
compute_te(
    pos=pos,
    vel=vel,
    TE_ver="ver4",
    TE_embedding=15,
    save_csv_path="empirical_data/TE_vals/fish_TElog_ver4_k15_full.csv"
)

In [ ]:
empirical_ver2_path = f"empirical_data/TE_vals/fish_TElog_ver2_k{TE_embedding}_full.csv"
empirical_ver4_path = f"empirical_data/TE_vals/fish_TElog_ver4_k{TE_embedding}_full.csv"

empirical_ver2 = pd.read_csv(empirical_ver2_path, index_col=0).mean(axis=1)
empirical_ver4 = pd.read_csv(empirical_ver4_path, index_col=0).mean(axis=1)


empirical_ver2 = empirical_ver2.rolling(window=15, center=True).mean()
empirical_ver4 = empirical_ver4.rolling(window=15, center=True).mean()

emp_ver2_x = np.arange(empirical_ver2_start, empirical_ver2_end)
emp_ver2_y = np.ravel(np.asarray(empirical_ver2))[empirical_ver2_start:empirical_ver2_end]

emp_ver4_x = np.arange(empirical_ver4_start, empirical_ver4_end)
emp_ver4_y = np.ravel(np.asarray(empirical_ver4))[empirical_ver4_start:empirical_ver4_end]

## Compute TE for D'Orsogna simulations

In [ ]:
# Compute TE for D'Orsogna simulations
ver2_results = []
ver4_results = []

for m in models:
    m["model"].compute_modelTE(TE_ver="ver2", TE_embedding=TE_embedding)
    m["model"].compute_modelTE(TE_ver="ver4", TE_embedding=TE_embedding)

    ver2_path = f"empirical_data/TE_vals/TElog_single_mill_{m['C']}_{m['l']}_ver2_k{TE_embedding}.csv"
    ver4_path = f"empirical_data/TE_vals/TElog_single_mill_{m['C']}_{m['l']}_ver4_k{TE_embedding}.csv"

    ver2_series = pd.read_csv(ver2_path, index_col=0).mean(axis=1)
    ver4_series = pd.read_csv(ver4_path, index_col=0).mean(axis=1)

    if smooth_window > 1:
        ver2_series = ver2_series.rolling(window=smooth_window, center=True).mean()
        ver4_series = ver4_series.rolling(window=smooth_window, center=True).mean()

    x2 = np.arange(m["t_start"], m["t_end"])
    y2 = np.ravel(np.asarray(ver2_series))[m["t_start"]:m["t_end"]]

    x4 = np.arange(m["t_start"], m["t_end"])
    y4 = np.ravel(np.asarray(ver4_series))[m["t_start"]:m["t_end"]]

    ver2_results.append({
        "label": m["label"],
        "x": x2,
        "y": y2
    })

    ver4_results.append({
        "label": m["label"],
        "x": x4,
        "y": y4
    })


## Visualize TE profiles

In [ ]:
# Same y-ranges within each version
all_ver2_y = [emp_ver2_y] + [res["y"] for res in ver2_results]
all_ver4_y = [emp_ver4_y] + [res["y"] for res in ver4_results]

ver2_ymin = np.nanmin([np.nanmin(y) for y in all_ver2_y])
ver2_ymax = np.nanmax([np.nanmax(y) for y in all_ver2_y])

ver4_ymin = np.nanmin([np.nanmin(y) for y in all_ver4_y])
ver4_ymax = np.nanmax([np.nanmax(y) for y in all_ver4_y])

### Version 2

In [ ]:
set_plot_style()

fig, axes = plt.subplots(2, 2, figsize=(10, 10))
axes = axes.flatten()

# top left: empirical
axes[0].plot(emp_ver2_x, emp_ver2_y, linewidth=3)
axes[0].set_title("Golden shiners")
axes[0].set_xlabel("Time step")
axes[0].set_ylabel("Average Local Transfer Entropy")
axes[0].axhline(0, color="gray", linewidth=0.6, linestyle=":")
axes[0].grid()
axes[0].set_ylim(ver2_ymin, ver2_ymax)

plt.axhline(0, color="gray", linewidth=0.6, linestyle=":")
plt.xlabel("Distance from Swarm Center ($r$)")
plt.ylabel("Averaged Tangential Velocity")
plt.title("Averaged Tangential Velocity Profile")

# other three: D'Orsogna simulations
for i in range(3):
    axes[i+1].plot(ver2_results[i]["x"], ver2_results[i]["y"], linewidth=3)
    axes[i+1].set_title(ver2_results[i]["label"])
    axes[i+1].set_xlabel("Time step")
    axes[i+1].set_ylabel("Average Local Transfer Entropy")
    axes[i+1].axhline(0, color="gray", linewidth=0.6, linestyle=":")
    axes[i+1].grid()
    axes[i+1].set_ylim(ver2_ymin, ver2_ymax)

formatter = FuncFormatter(lambda x, pos: f"{x/10:g}")
for ax in axes:
    ax.xaxis.set_major_formatter(formatter)

plt.tight_layout()

folder_name = "empirical_graphs"
if not os.path.exists(folder_name):
    os.makedirs(folder_name)

plt.savefig(os.path.join(folder_name, "te_ver2_grid.png"), dpi=300, bbox_inches="tight")
plt.show()

### Version 4

In [ ]:
set_plot_style()

# top left: empirical
axes[0].plot(emp_ver4_x, emp_ver4_y, linewidth=3)
axes[0].set_title("Golden shiners")
axes[0].set_xlabel("Time step")
axes[0].set_ylabel("Average Local Transfer Entropy")
axes[0].axhline(0, color="gray", linewidth=0.6, linestyle=":")
axes[0].grid()
axes[0].set_ylim(ver4_ymin, ver4_ymax)

# other three: D'Orsogna simulations
for i in range(3):
    axes[i+1].plot(ver4_results[i]["x"], ver4_results[i]["y"], linewidth=3)
    axes[i+1].set_title(ver4_results[i]["label"])
    axes[i+1].set_xlabel("Time step")
    axes[i+1].set_ylabel("Average Local Transfer Entropy")
    axes[i+1].axhline(0, color="gray", linewidth=0.6, linestyle=":")
    axes[i+1].grid()
    axes[i+1].set_ylim(ver4_ymin, ver4_ymax)

formatter = FuncFormatter(lambda x, pos: f"{x/10:g}")
for ax in axes:
    ax.xaxis.set_major_formatter(formatter)

plt.tight_layout()
plt.savefig(os.path.join(folder_name, "te_ver4_grid.png"), dpi=300, bbox_inches="tight")
plt.show()